# Threshold Tuning (Harness Configuration)

Find the optimal quality threshold that balances research quality against token cost and latency.

## Learning Objectives
- Understand the quality-cost tradeoff
- Analyze iteration patterns at different thresholds
- Configure the harness quality threshold for your use case

In [ ]:
import os
import sys
import httpx
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
env_path = os.path.join('..', '.env')
load_dotenv(env_path, override=True)

_VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")
_http_client = None if _VERIFY_SSL else httpx.Client(verify=False, timeout=httpx.Timeout(300.0))

from harness.metrics import MetricsAggregator
print("Metrics module loaded")

## 1. Quality vs Cost Tradeoff

| Threshold | Typical Iterations | Token Cost | Quality |
|-----------|-------------------|------------|----------|
| 5.0 (Low) | 1-2 | ~5K tokens | Basic |
| 7.0 (Medium) | 2-3 | ~12K tokens | Good |
| 9.0 (High) | 3-5 | ~25K tokens | Excellent |

In [ ]:
# Simulate metrics for different threshold scenarios
scenarios = {
    "Low (5.0)": [
        {"iteration": 1, "tokens": 4800, "quality": 5.5, "passed": True},
    ],
    "Medium (7.0)": [
        {"iteration": 1, "tokens": 5200, "quality": 4.5, "passed": False},
        {"iteration": 2, "tokens": 5800, "quality": 7.2, "passed": True},
    ],
    "High (9.0)": [
        {"iteration": 1, "tokens": 5200, "quality": 4.5, "passed": False},
        {"iteration": 2, "tokens": 5800, "quality": 6.8, "passed": False},
        {"iteration": 3, "tokens": 6200, "quality": 8.2, "passed": False},
        {"iteration": 4, "tokens": 5500, "quality": 9.1, "passed": True},
    ],
}

print(f"{'Threshold':<15} {'Iterations':<12} {'Total Tokens':<14} {'Final Quality':<14} {'Cost (est)'}")
print("-" * 65)
for name, iterations in scenarios.items():
    total_tokens = sum(it['tokens'] for it in iterations)
    final_quality = iterations[-1]['quality']
    est_cost = total_tokens * 0.5 / 1_000_000  # Rough estimate
    print(f"{name:<15} {len(iterations):<12} {total_tokens:<14} {final_quality:<14.1f} ${est_cost:.4f}")

## 2. Recommended Configuration

For most research tasks, the **Medium (7.0)** threshold provides the best balance:
- Usually completes in 2-3 iterations
- Quality sufficient for research reports
- Reasonable token cost

Configure in `.env`:
```
QUALITY_THRESHOLD=7.0
MAX_ITERATIONS=3
```

In [ ]:
# Current configuration
print("Current settings:")
print(f"  QUALITY_THRESHOLD: {os.getenv('QUALITY_THRESHOLD', '7.0')}")
print(f"  MAX_ITERATIONS: {os.getenv('MAX_ITERATIONS', '3')}")
print(f"  LLM_MODEL: {os.getenv('LLM_MODEL', 'granite-3.3-8b-instruct')}")
print("\nAdjust these in .env based on your quality/cost requirements.")

## Summary

Threshold tuning guidelines:
- **Start with 7.0** for a balanced approach
- **Lower to 5.0** for quick, exploratory research
- **Raise to 8.5+** for critical, publication-quality research
- **Always set max_iterations** as a safety limit (3-5 recommended)